In [1]:
import torch; torch.manual_seed(0)
import torch.nn as nn
import torch.nn.functional as F
import torch.utils
import torch.distributions
import torchvision
import numpy as np
import matplotlib.pyplot as plt; plt.rcParams['figure.dpi'] = 200
from afqinsight import AFQDataset
from afqinsight.nn.utils import prep_pytorch_data
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from torch.distributions.normal import Normal
from sklearn.decomposition import PCA
import afqinsight.augmentation as aug
from afqinsight.nn.pt_models import Conv1DAutoencoder
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
import sys 
sys.path.insert(1, '/Users/samchou/AFQ-Insight-Autoencoder-Plotting/Experiment_Utils')
# sys.path.insert(1, '/mmfs1/gscratch/nrdg/samchou/AFQ-Insight-Autoencoder-Experiments/Experiment_Utils')
from utils import select_device, prep_fa_dataset, prep_first_tract_data, prep_fa_flattned_data, prep_fa_flattened_remapped_data, GradReverse
from models import Conv1DVariationalAutoencoder_fa, AgePredictorCNN, SitePredictorCNN, CombinedVAE_Predictors

In [3]:
device = select_device()

Using device: mps

Using MPS backend on macOS. (Detailed memory info may not be available.)


In [4]:
dataset = AFQDataset.from_study('hbn')
torch_dataset, train_loader, test_loader, val_loader = prep_pytorch_data(dataset,batch_size=128)  
gt_shape = torch_dataset[0][1].size()[0]
sequence_length = torch_dataset[0][0].size()[0]  # 48
in_channels = torch_dataset[0][0].size()[1]  # 100


File /Users/samchou/.cache/afq-insight/hbn/subjects.tsv exists.
File /Users/samchou/.cache/afq-insight/hbn/nodes.csv exists.


/Users/samchou/src/nrdg/AFQ-Insight/afqinsight/transform.py:144: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  features = interpolated.stack(["subjectID", "tractID", "metric"]).unstack(


In [5]:
site_idx = dataset.target_cols.index('scan_site_id')
age_idx = dataset.target_cols.index('age')

In [6]:
unique_sites = np.unique(torch_dataset.y[:, site_idx])
print(f"Unique site values found: {unique_sites}")
site_map = {float(site): i for i, site in enumerate(sorted(unique_sites))}
print(f"Created site map: {site_map}")
num_sites = len(site_map)
print(f"Number of unique sites (classes): {num_sites}")


Unique site values found: [0. 1. 3. 4.]
Created site map: {0.0: 0, 1.0: 1, 3.0: 2, 4.0: 3}
Number of unique sites (classes): 4


In [7]:
def extract_first_tract_data(dataset, batch_size=32):
    """
    Custom function to extract first tract data from a PyTorch dataset,
    without calling prep_pytorch_data again.

    Parameters
    ----------
    dataset : torch.utils.data.Dataset or tuple
        Dataset containing tract data or tuple of (dataset, train_loader, test_loader, val_loader)
    batch_size : int
        Batch size for data loaders

    Returns
    -------
    tuple
        First tract train, test, and validation loaders
    """
    print("DEBUG: Creating first tract dataset")
    
    class FirstTractDataset(torch.utils.data.Dataset):
        def __init__(self, original_dataset):
            self.original_dataset = original_dataset

        def __len__(self):
            return len(self.original_dataset)

        def __getitem__(self, idx):
            x, y = self.original_dataset[idx]
            # Extract just the first tract (tract 0)
            tract_data = x[0:1, :].clone()
            return tract_data, y

    # Handle different dataset formats
    if isinstance(dataset, tuple) and len(dataset) == 4:
        # If dataset is a tuple returned from prep_fa_dataset, unpack it
        torch_dataset, train_loader, test_loader, val_loader = dataset
        
        # Extract datasets from loaders
        train_dataset = train_loader.dataset
        test_dataset = test_loader.dataset
        val_dataset = val_loader.dataset
        
        print(f"DEBUG: Using datasets from data loaders, train dataset size: {len(train_dataset)}")
    elif hasattr(dataset, 'train_data') and hasattr(dataset, 'test_data') and hasattr(dataset, 'val_data'):
        # If dataset has train/test/val data attributes
        train_dataset = dataset.train_data
        test_dataset = dataset.test_data
        val_dataset = dataset.val_data
        print(f"DEBUG: Using train/test/val from dataset attributes, train dataset size: {len(train_dataset)}")
    else:
        # If dataset is a single dataset, use it for all (not ideal but fallback)
        print(f"DEBUG: Unknown dataset structure, attempting to use directly. Type: {type(dataset)}")
        if hasattr(dataset, '__getitem__') and hasattr(dataset, '__len__'):
            train_dataset = dataset
            test_dataset = dataset
            val_dataset = dataset
            print(f"DEBUG: Using dataset directly, size: {len(dataset)}")
        else:
            raise ValueError(f"Unsupported dataset type: {type(dataset)}. Cannot extract first tract.")

    # Create first tract datasets
    first_tract_train = FirstTractDataset(train_dataset)
    first_tract_test = FirstTractDataset(test_dataset)
    first_tract_val = FirstTractDataset(val_dataset)

    # Create data loaders
    first_tract_train_loader = torch.utils.data.DataLoader(
        first_tract_train, batch_size=batch_size, shuffle=True
    )
    first_tract_test_loader = torch.utils.data.DataLoader(
        first_tract_test, batch_size=batch_size, shuffle=False
    )
    first_tract_val_loader = torch.utils.data.DataLoader(
        first_tract_val, batch_size=batch_size, shuffle=False
    )
    
    print(f"DEBUG: Created first tract data loaders. Train size: {len(first_tract_train)}")
    return first_tract_train_loader, first_tract_test_loader, first_tract_val_loader

In [8]:
fa_md_output = prep_fa_dataset(dataset, target_labels=["dki_fa", "dki_md"], batch_size=128)
print(f"DEBUG: Got FA+MD dataset, type: {type(fa_md_output)}")
if isinstance(fa_md_output, tuple) and len(fa_md_output) == 4:
    print("DEBUG: Dataset is a tuple of (dataset, train_loader, test_loader, val_loader)")
else:
    print(f"DEBUG: Dataset is of type {type(fa_md_output)}")
sys.stdout.flush()

# Now extract just the first tract from this dataset
print("DEBUG: Extracting first tract data")
sys.stdout.flush()
first_tract_train_loader, first_tract_test_loader, first_tract_val_loader = extract_first_tract_data(
    fa_md_output, batch_size=128
)

DEBUG: Got FA+MD dataset, type: <class 'tuple'>
DEBUG: Dataset is a tuple of (dataset, train_loader, test_loader, val_loader)
DEBUG: Extracting first tract data
DEBUG: Creating first tract dataset
DEBUG: Using datasets from data loaders, train dataset size: 1194
DEBUG: Created first tract data loaders. Train size: 1194


In [9]:
print("Preparing initial PyTorch data loaders...")
try:
    # Assuming prep_pytorch_data returns torch_dataset, train_loader, test_loader, val_loader
    # If it returns datasets, create loaders here.
    # Adapt this call based on the actual signature and return values of your prep_pytorch_data
    prep_output = prep_fa_flattened_remapped_data(dataset, batch_size=128)
    if len(prep_output) == 4:
        _, train_loader_raw, test_loader_raw, val_loader_raw = prep_output
    else:
        raise ValueError(f"Expected 4 return values from prep_pytorch_data, got {len(prep_output)}")

    print("Initial data loaders prepared.")
except Exception as e:
     print(f"Error calling prep_pytorch_data: {e}")
     print("Ensure the function exists and returns DataLoaders or required components.")
     sys.exit(1)

Preparing initial PyTorch data loaders...
Remapping prep: Using age index 0, site index 2 from ['age', 'sex', 'scan_site_id']
Using site map: {0.0: 0.0, 1.0: 1.0, 3.0: 2.0, 4.0: 3.0}
Creating remapped datasets...
Creating final DataLoaders...
prep_fa_flattened_remapped_data complete.
Initial data loaders prepared.


In [ ]:
latent_dim = 64  # Same latent dimension as the saved model
dropout = 0.0    # Same dropout rate as the saved model

# Create a new model instance
model = Conv1DVariationalAutoencoder_fa(latent_dims=latent_dim, dropout=dropout, input_length=100).to(device)

# Load weights - adjust file path as needed
# If loading on CPU from a model trained on GPU, use map_location
model.load_state_dict(torch.load('model_weights/best_vae.pth', map_location=device))

# Set model to evaluation mode for inference
model.eval()

/var/folders/gc/z2m2_nxj4qlf83r8bq6hz_240000gn/T/ipykernel_20295/286970393.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('model_weight

RuntimeError: Error(s) in loading state_dict for Conv1DVariationalAutoencoder_fa:
	Missing key(s) in state_dict: "encoder.conv2.weight", "encoder.conv2.bias", "decoder.deconv3.weight", "decoder.deconv3.bias". 
	Unexpected key(s) in state_dict: "encoder.conv2_50.weight", "encoder.conv2_50.bias", "encoder.conv2_100.weight", "encoder.conv2_100.bias", "decoder.deconv3_100.weight", "decoder.deconv3_100.bias", "decoder.deconv3_50.weight", "decoder.deconv3_50.bias". 

In [ ]:
# Example of using the loaded model for reconstruction
with torch.no_grad():  # No gradient tracking needed for inference
    # Prepare your test sample
    test_sample = first_tract_test_loader.dataset[0][0][0:1].unsqueeze(0).to(device)  # Shape should be (batch_size, 1, sequence_length)
    
    # Get reconstruction
    reconstructed, mean, logvar = model(test_sample)
    
    # Now you can compare the original and reconstructed data
    # For example, plotting them
    import matplotlib.pyplot as plt
    
    # For a single example in the batch
    original = test_sample[0, 0].cpu().numpy()
    recon = reconstructed[0, 0].cpu().numpy()
    
    plt.figure(figsize=(10, 5))
    plt.plot(original, label='Original')
    plt.plot(recon, label='Reconstructed')
    plt.legend()
    plt.title('Original vs Reconstructed Signal')
    plt.show()

In [ ]:
sample = first_tract_test_loader.dataset[0][0][0:1].unsqueeze(0).to(device)
output = model(sample)

# Assuming the first element of the tuple is the reconstruction:
reconstructed = output[0]

orig = sample.cpu().detach().numpy()
recon = reconstructed.cpu().detach().numpy()

plt.plot(orig.flatten()[0:100], color='blue', label='Original')
plt.plot(recon.flatten()[0:100], color='red', label='Reconstructed')